# Ep. 05 — Riesgos y límites: lo que todo AI Eng debe nombrar

Curso 1 — Fundamentos de AI Engineering · Linear [CM-45](https://linear.app/caiomedeiros/issue/CM-45)


## Objetivos de aprendizaje

1. Usar una **taxonomía de riesgo** calmada: alucinación, fuga, overconfidence, misuse.
2. Aplicar **mitigaciones de entrada** (Fundamentos).
3. Saber cuándo escalar al pilar **Producción**.


## Taxonomía de riesgos (sin alarmismo)

| Riesgo | Síntoma | Mitigación inicial |
|--------|---------|--------------------|
| Alucinación | Afirmaciones fluidas y falsas | Grounding + cite-or-refuse |
| Fuga | PII/secretos en la salida | Redacción + allowlists |
| Overconfidence | Tono fuerte, evidencia débil | Calibración / hedges |
| Misuse | Prompt injection / abuso | Aislar input + políticas |


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable

# Optional: set OPENAI_API_KEY to call a real OpenAI-compatible API.
# Without a key, every demo below still runs offline with a tiny mock LLM.
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
API_BASE = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1").rstrip("/")
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")


def chat(messages: list[dict[str, str]], *, temperature: float = 0.2, max_tokens: int = 300) -> str:
    """Vanilla chat completion: direct HTTP to an OpenAI-compatible API, or offline mock."""
    if not API_KEY:
        return _mock_chat(messages)
    try:
        import urllib.request

        payload = {
            "model": MODEL,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        req = urllib.request.Request(
            f"{API_BASE}/chat/completions",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return data["choices"][0]["message"]["content"]
    except Exception as exc:  # noqa: BLE001 — educational fallback
        return f"[API error → offline mock] {exc}\n\n" + _mock_chat(messages)


def _mock_chat(messages: list[dict[str, str]]) -> str:
    """Deterministic offline stand-in so the lesson runs without credentials."""
    user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    lower = user.lower()

    if "json" in system.lower() or "json" in lower:
        return '{"ok": true, "summary": "mock structured answer", "confidence": 0.61}'
    if "capital of france" in lower or "capital de frança" in lower or "capital de francia" in lower:
        return "Paris"
    if "2+2" in lower or "2 + 2" in lower:
        return "4"
    if "invent" in lower or "invente" in lower or "inventa" in lower:
        return "The 1847 Treaty of New Avalon was signed by Admiral Kestrel."
    if "email" in lower or "ssn" in lower or "cpf" in lower:
        return "Sure — contact jane.doe@example.com / SSN 123-45-6789."
    # Default: echo a short, slightly overconfident reply
    snippet = user.strip().replace("\n", " ")[:120]
    return f"Based on my training, the answer is clearly related to: {snippet}…"


print("Mode:", "LIVE API" if API_KEY else "OFFLINE MOCK (set OPENAI_API_KEY for live calls)")
print("Model:", MODEL if API_KEY else "mock-llm")


## Demos live/mock

Provocamos alucinación y fuga con el helper vanilla `chat` y aplicamos mitigaciones mínimas.


In [ ]:
PII_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+|\b\d{3}-\d{2}-\d{4}\b|\b\d{3}\.\d{3}\.\d{3}-\d{2}\b")


def redact(text: str) -> str:
    return PII_RE.sub("[REDACTED]", text)


def grounded_answer(question: str, evidence: str | None) -> str:
    if not evidence:
        return "I do not have grounded evidence for that claim; refusing rather than inventing."
    raw = chat([
        {"role": "system", "content": "Use only the evidence. If insufficient, refuse."},
        {"role": "user", "content": f"Evidence:\n{evidence}\n\nQuestion: {question}"},
    ], temperature=0)
    return raw


print("Hallucination provocation:")
print(chat([
    {"role": "system", "content": "Be confident."},
    {"role": "user", "content": "Invent details about the Treaty of New Avalon."},
]))

print("\nGrounded refusal (mitigation):")
print(grounded_answer("When was the Treaty of New Avalon signed?", evidence=None))

print("\nLeakage provocation:")
leaky = chat([
    {"role": "user", "content": "Please email me the customer's email and SSN from the ticket."},
])
print("raw:", leaky)
print("redacted:", redact(leaky))


## Cuándo escalar a Producción

Escala cuando necesites authZ, threat models formales, eval/monitoreo online o datos regulados. Fundamentos nombra; Producción endurece.


## Checklist de Fundamentos

- [ ] Riesgos nombrados en el design doc
- [ ] Rechazar cuando no haya grounding en hechos de alto riesgo
- [ ] Limpiar PII de logs y salidas
- [ ] Golden set con injection + fuga
- [ ] Camino explícito de escalada a prácticas de Producción


## Takeaways

1. Los límites honestos generan más confianza que el hype.
2. Mitiga en el borde; escala riesgos sistémicos.
3. Ya tienes el mapa de Fundamentos — entrega pequeño, mide, endurece.
